# Problemas de Engenharia Resolvidos com Inteligência Artificial

## 1. Previsão da Conversão de Reação Catalítica

Em um processo industrial de conversão de etileno ($C_2H_4$) em óxido de etileno ($C_2H_4O$), a eficiência do reator catalítico depende da __temperatura__ de operação e da __concentração__ inicial de etileno na corrente de entrada.

Com base em dados experimentais obtidos em planta piloto, deseja-se prever a conversão ($X$) da reação (em fração, de $0$ a $1$) a partir da Temperatura $T$ (em $^oC$) e da Concentração inicial $C_o$ (em mol/L). Para isso, uma __Rede Neural__ com uma camada oculta e __Função de Ativação Sigmoide__ será utilizada.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Funções de ativação
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def deriv_sigmoid(x):
    fx = sigmoid(x)
    return fx * (1 - fx)

def mse_loss(y_true, y_pred):
    return ((y_true - y_pred) ** 2).mean()

# Classe da Rede Neural
class NeuralNetwork:
    def __init__(self):
        # Pesos e bias para camada oculta (4 neurônios)
        self.w = np.random.normal(size=(4, 2))  # 4 neurônios, 2 entradas (T e C)
        self.b = np.random.normal(size=4)

        # Pesos e bias para camada de saída (1 neurônio)
        self.w_out = np.random.normal(size=4)  # um peso por neurônio oculto
        self.b_out = np.random.normal()

        # Histórico da perda
        self.loss_history = []

    def feedforward(self, x):
        h_sum = np.dot(self.w, x) + self.b
        h_out = sigmoid(h_sum)

        o_sum = np.dot(self.w_out, h_out) + self.b_out
        o_out = sigmoid(o_sum)

        return o_out, h_out, h_sum, o_sum

    def train(self, data, targets, epochs=1000, learn_rate=0.1):
        for epoch in range(epochs):
            for x, y_true in zip(data, targets):
                x = np.array(x)
                y_true = float(y_true)

                # Forward
                y_pred, h_out, h_sum, o_sum = self.feedforward(x)

                # Derivadas
                dL_dypred = -2 * (y_true - y_pred)
                dypred_do_sum = deriv_sigmoid(o_sum)

                dL_dw_out = dL_dypred * dypred_do_sum * h_out
                dL_db_out = dL_dypred * dypred_do_sum

                dypred_dh = self.w_out * dypred_do_sum
                dL_dh = dL_dypred * dypred_do_sum * deriv_sigmoid(h_sum)

                dL_dw = np.outer(dL_dh, x)
                dL_db = dL_dh

                # Atualização dos pesos
                self.w_out -= learn_rate * dL_dw_out
                self.b_out -= learn_rate * dL_db_out

                self.w -= learn_rate * dL_dw
                self.b -= learn_rate * dL_db

            # Registrar a perda da época
            y_preds = np.array([self.feedforward(x)[0] for x in data])
            loss = mse_loss(targets, y_preds)
            self.loss_history.append(loss)

            if epoch % 100 == 0:
                print(f'Epoch {epoch} Loss: {loss:.4f}')

    def predict(self, x):
        y_pred, _, _, _ = self.feedforward(np.array(x))
        return y_pred

    def plot_loss(self):
        plt.figure(figsize=(8,5))
        plt.plot(self.loss_history, label='Erro quadrático médio (MSE)')
        plt.xlabel('Épocas')
        plt.ylabel('Perda')
        plt.title('Evolução da Perda durante o Treinamento')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

# Dados (Temperatura em °C, Concentração em mol/L)
data = np.array([
    [200, 1.0],
    [250, 1.2],
    [300, 1.5],
    [350, 1.8],
    [400, 2.0]
])

# Conversão (X) associada aos dados
targets = np.array([0.10, 0.35, 0.60, 0.80, 0.95])

# Treinamento
nn = NeuralNetwork()
nn.train(data, targets, epochs=1000, learn_rate=0.1)

# Gráfico da perda
nn.plot_loss()

# Novas condições operacionais
novos_dados = [
    [275, 1.4],
    [375, 1.9],
    [220, 1.1],
    [310, 1.6]
]

print('Predições para novas condições operacionais:')
for entrada in novos_dados:
    x = entrada
    y_pred = nn.predict(x)
    print(f'T = {x[0]} °C, C₀ = {x[1]:.2f} mol/L → X ≈ {y_pred:.3f}')

## 2. Previsão da Composição em uma Corrente de Destilação

Uma planta de destilação de etanol coleta dados operacionais de várias correntes do processo para controlar a qualidade do produto. Os engenheiros estão interessados em prever a fração mássica de etanol em diferentes correntes, com base em variáveis de processo como temperatura e vazão mássica.

In [ ]:
import numpy as np

# Função de normalização mín-máx
def normalizar(dados):
    minimo = dados.min(axis=0)
    maximo = dados.max(axis=0)
    return (dados - minimo) / (maximo - minimo), minimo, maximo

# Dados de entrada: Temperatura (°C), Vazão (kg/h)
entradas = np.array([
    [78, 1200],  # Corrente A
    [72, 1350],  # Corrente B
    [80, 1100],  # Corrente C
    [70, 1250]  # Corrente D
], dtype=np.float32)

# Dados de saída: Fração de etanol (%)
saidas = np.array([
    [95],
    [88],
    [97],
    [85]
], dtype=np.float32)

# Normalizando entradas e saídas
entradas_norm, entradas_min, entradas_max = normalizar(entradas)
saidas_norm, saidas_min, saidas_max = normalizar(saidas)

# Função ReLU e derivada
def relu(x):
    return np.maximum(0, x)

def relu_derivada(x):
    return (x> 0).astype(float)

# Inicialização dos pesos.
np.random.seed(42)

# 2 entradas, 3 neurônios na camada oculta, 1 saída.
pesos_entrada_oculta = np.random.randn(2,3)
bias_oculta = np.zeros((1,3))

pesos_oculta_saida = np.random.randn(3, 1)
bias_saida = np.zeros((1,1))

#Hiperparâmetros
taxa_aprendizado = 0.1
epochs = 1000

for epoca in range(epochs):
    # FORWARD
    camada_oculta_input = np.dot(entradas_norm, pesos_entrada_oculta) + bias_oculta
    camada_oculta_saida = relu(camada_oculta_input)

    saida_input = np.dot(camada_oculta_saida, pesos_oculta_saida) + bias_saida
    saida_predita = saida_input  # Sem função de ativação final (regressão)

    # ERRO
    erro = saidas_norm - saida_predita

    # BACKPROPAGATION
    d_saida = -2 * erro  # derivada MSE

    d_pesos_oculta_saida = np.dot(camada_oculta_saida.T, d_saida)
    d_bias_saida = d_saida.sum(axis=0, keepdims=True)

    d_oculta = np.dot(d_saida, pesos_oculta_saida.T) * relu_derivada(camada_oculta_input)
    d_pesos_entrada_oculta = np.dot(entradas_norm.T, d_oculta)
    d_bias_oculta = d_oculta.sum(axis=0, keepdims=True)

    # ATUALIZAÇÃO
    pesos_oculta_saida -= taxa_aprendizado * d_pesos_oculta_saida
    bias_saida -= taxa_aprendizado * d_bias_saida

    pesos_entrada_oculta -= taxa_aprendizado * d_pesos_entrada_oculta
    bias_oculta -= taxa_aprendizado * d_bias_oculta

    # Exibir erro médio a cada 200 épocas
    if epoca % 200 == 0:
        mse = np.mean(erro**2)
        print(f'Época {epoca}, Erro quadrático médio: {mse:.4f}')

# Novas entradas
novas_entradas = np.array([
    [75, 1300],
    [73, 1180]
], dtype=np.float32)

# Normalizar novas entradas com base nos dados originais
novas_entradas_norm = (novas_entradas - entradas_min) / (entradas_max - entradas_min)

# Forward para previsão
camada_oculta_nova = relu(np.dot(novas_entradas_norm, pesos_entrada_oculta) + bias_oculta)
saida_nova_norm = np.dot(camada_oculta_nova, pesos_oculta_saida) + bias_saida

# Desnormalizar saída
saida_nova = saida_nova_norm * (saidas_max - saidas_min) + saidas_min
print('Previsões para novas correntes:')
for i, pred in enumerate(saida_nova):
    print(f'Corrente {i+1}: Fração de etanol estimada = {pred[0]:.2f}%')

## 3. Classificação do Tipo de Reator

Em uma planta de processos químicos, diferentes reatores são utilizados conforme as características da reação, como a necessidade de agitação contínua, regime de operação ou tempo de residência. Os dados operacionais mais comuns utilizados no projeto e na escolha desses reatores incluem a __massa de catalisador__ e a __temperatura__ de operação.

O objetivo é treinar uma rede neural que consiga classificar o tipo de reator com base nessas duas variáveis operacionais.

In [ ]:
import numpy as np

# Dados de entrada: [massa (g), temperatura (°C)]
entradas = np.array([
    [250, 320],  # R-101
    [300, 350],  # R-102
    [275, 340],  # R-103
    [225, 310]  # R-104
], dtype=np.float32)

# Saída (one-hot encoding):
# CSTR = [1, 0, 0], PFR = [0, 1, 0], BATCH = [0, 0, 1]
saidas = np.array([
    [1, 0, 0],  # R-101 → CSTR
    [0, 1, 0],  # R-102 → PFR
    [1, 0, 0],  # R-103 → CSTR
    [0, 0, 1]  # R-104 → BATCH
], dtype=np.float32)

def normalizar(dados):
    minimo = dados.min(axis=0)
    maximo = dados.max(axis=0)
    return (dados - minimo) / (maximo - minimo), minimo, maximo

entradas_norm, entradas_min, entradas_max = normalizar(entradas)

def relu(x):
    return np.maximum(0, x)

def relu_derivada(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))  # estabilidade numérica
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

np.random.seed(0)

# Estrutura: 2 entradas, 4 neurônios ocultos, 3 saídas
pesos_entrada_oculta = np.random.randn(2, 4)
bias_oculta = np.zeros((1, 4))

pesos_oculta_saida = np.random.randn(4, 3)
bias_saida = np.zeros((1, 3))

taxa_aprendizado = 0.1
epochs = 1000

for epoca in range(epochs):
    # Forward
    z_oculta = np.dot(entradas_norm, pesos_entrada_oculta) + bias_oculta
    a_oculta = relu(z_oculta)

    z_saida = np.dot(a_oculta, pesos_oculta_saida) + bias_saida
    saida_predita = softmax(z_saida)

    # Erro (categorical cross-entropy simplificada)
    erro = saidas - saida_predita

    # Backpropagation
    d_saida = erro  # derivada direta da cross-entropy com softmax

    d_pesos_oculta_saida = np.dot(a_oculta.T, d_saida)
    d_bias_saida = np.sum(d_saida, axis=0, keepdims=True)

    d_oculta = np.dot(d_saida, pesos_oculta_saida.T) * relu_derivada(z_oculta)
    d_pesos_entrada_oculta = np.dot(entradas_norm.T, d_oculta)
    d_bias_oculta = np.sum(d_oculta, axis=0, keepdims=True)

    # Atualização dos pesos e bias
    pesos_oculta_saida += taxa_aprendizado * d_pesos_oculta_saida
    bias_saida += taxa_aprendizado * d_bias_saida

    pesos_entrada_oculta += taxa_aprendizado * d_pesos_entrada_oculta
    bias_oculta += taxa_aprendizado * d_bias_oculta

    # Erro médio
    if epoca % 200 == 0:
        loss = np.mean(np.square(erro))
        print(f'Época {epoca} - Erro quadrático médio: {loss:.4f}')

# Novas entradas: [massa, temperatura]
novos = np.array([
    [260, 330],
    [290, 345]
], dtype=np.float32)

# Normalizar com base no conjunto original
novos_norm = (novos - entradas_min) / (entradas_max - entradas_min)

# Forward para novas previsões
z_oculta_novos = np.dot(novos_norm, pesos_entrada_oculta) + bias_oculta
a_oculta_novos = relu(z_oculta_novos)

z_saida_novos = np.dot(a_oculta_novos, pesos_oculta_saida) + bias_saida
saida_novos = softmax(z_saida_novos)

# Interpretar as previsões
tipos = ['CSTR', 'PFR', 'BATCH']
print('Previsões para novos reatores:')
for i, pred in enumerate(saida_novos):
    tipo_previsto = tipos[np.argmax(pred)]
    print(f'Reator {i+1}: tipo previsto = {tipo_previsto} (distribuição: {pred.round(3)})')

## 4. Previsão da Eficiência de Conversão de um Reator Catalítico

Um Engenheiro Químico deseja prever a __eficiência de conversão de um reator catalítico__ ($\eta$, variando de 0 a 1) a partir de três variáveis de processo:

- $T$: Temperatura de operação (em $^oC$)
- $C_A$: Concentração inicial do reagente A (mol/L)
- $P$: Pressão de operação (bar)

Foi proposta uma rede neural __feedforward__ com 3 camadas ocultas para prever a eficiência.

In [ ]:
import numpy as np

# Função sigmoide
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# =========================
# Entrada
# =========================
x1, x2, x3 = 350, 1.2, 5.0
entrada = np.array([[x1, x2, x3]])

# =========================
# Pesos e Bias - Camada Oculta 1
# =========================
w11, w12, w13 = 0.12, -0.25, 0.33
w21, w22, w23 = -0.45, 0.67, -0.12
w31, w32, w33 = 0.56, 0.11, -0.78

b1_1, b1_2, b1_3 = 0.10, -0.05, 0.20

W1 = np.array([[w11, w12, w13],
               [w21, w22, w23],
               [w31, w32, w33]])
B1 = np.array([[b1_1, b1_2, b1_3]])

# =========================
# Pesos e Bias - Camada Oculta 2
# =========================
w11_2, w12_2, w13_2 = -0.32, 0.25, 0.40
w21_2, w22_2, w23_2 = 0.28, -0.14, 0.33
w31_2, w32_2, w33_2 = -0.56, 0.22, 0.15

b2_1, b2_2, b2_3 = -0.10, 0.05, -0.15

W2 = np.array([[w11_2, w12_2, w13_2],
               [w21_2, w22_2, w23_2],
               [w31_2, w32_2, w33_2]])
B2 = np.array([[b2_1, b2_2, b2_3]])

# =========================
# Pesos e Bias - Camada Oculta 3
# =========================
w11_3, w12_3, w13_3 = 0.45, -0.20, 0.30
w21_3, w22_3, w23_3 = -0.25, 0.60, -0.50
w31_3, w32_3, w33_3 = 0.15, -0.40, 0.55

b3_1, b3_2, b3_3 = 0.05, -0.05, 0.10

W3 = np.array([[w11_3, w12_3, w13_3],
               [w21_3, w22_3, w23_3],
               [w31_3, w32_3, w33_3]])
B3 = np.array([[b3_1, b3_2, b3_3]])

# =========================
# Pesos e Bias - Camada de Saída
# =========================
w11_4, w21_4, w31_4 = 0.40, -0.30, 0.25

b4_1 = 0.05

W4 = np.array([[w11_4],
               [w21_4],
               [w31_4]])
B4 = np.array([[b4_1]])

# =========================
# Forward Pass
# =========================
# Camada oculta 1
z1 = np.dot(entrada, W1) + B1
a1 = sigmoid(z1)

# Camada oculta 2
z2 = np.dot(a1, W2) + B2
a2 = sigmoid(z2)

# Camada oculta 3
z3 = np.dot(a2, W3) + B3
a3 = sigmoid(z3)

# Saída
z4 = np.dot(a3, W4) + B4
saida = sigmoid(z4)

# =========================
# Resultado
# =========================
print(f'Eficiência prevista do reator: {saida[0,0]:.4f}')